# Fuel-CSP — Comparative Analysis Notebook

This notebook re-runs the experiment matrix and renders every plot inline. It mirrors `scripts/run_experiments.py` and `scripts/generate_report.py`.

## 1. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from fuel_csp.analyzer import ExperimentConfig, run_matrix, save_csvs
from fuel_csp.visualizer import (
    plot_runtime, plot_nodes, plot_backtracks, plot_objective,
    plot_failure_rate, plot_heuristic_bars,
    plot_min_conflicts_convergence, plot_problem_topology,
)
from fuel_csp.synthetic import GeneratorConfig, generate_problem
from fuel_csp.algorithms import ALL_SOLVERS


## 2. Run the experiment matrix

We sweep N ∈ {10, 20, 30, 40, 50} and 5 random seeds per cell.

In [ ]:
cfg = ExperimentConfig(sizes=(10, 20, 30, 40, 50),
                       seeds=(7, 13, 21, 42, 99),
                       time_budget_s=6.0)
df = run_matrix(cfg)
df.head()


## 3. Per-(algorithm × N) summary

In [ ]:
from fuel_csp.analyzer import summarise
summary = summarise(df)
summary


## 4. Plots — scalability

In [ ]:
import pathlib
out = pathlib.Path('../results/plots')
out.mkdir(parents=True, exist_ok=True)
plot_runtime(df, out / 'runtime_vs_n.png');
plot_nodes(df, out / 'nodes_vs_n.png');
plot_backtracks(df, out / 'backtracks_vs_n.png');
from IPython.display import Image, display
for f in ['runtime_vs_n.png', 'nodes_vs_n.png', 'backtracks_vs_n.png']:
    display(Image(out / f))


## 5. Plots — solution quality + failure rate

In [ ]:
plot_objective(df, out / 'objective_vs_n.png');
plot_failure_rate(df, out / 'failure_rate_vs_n.png');
plot_heuristic_bars(df, out / 'heuristic_bars.png');
for f in ['objective_vs_n.png', 'failure_rate_vs_n.png',
          'heuristic_bars.png']:
    display(Image(out / f))


## 6. Min-Conflicts convergence

In [ ]:
from fuel_csp.analyzer import run_one
traces = [run_one('min_conflicts', 30, s, cfg).stats.cost_trace
          for s in cfg.seeds]
plot_min_conflicts_convergence(traces, out / 'min_conflicts_convergence.png');
display(Image(out / 'min_conflicts_convergence.png'))


## 7. Sample instance topology

In [ ]:
problem = generate_problem(GeneratorConfig(num_vehicles=30, seed=7))
solver = ALL_SOLVERS['bt_fc_mrv_deg'](time_budget_s=6.0)
res = solver.solve(problem)
plot_problem_topology(problem, res.assignment, out / 'sample_topology.png');
display(Image(out / 'sample_topology.png'))


## 8. Take-aways

* Basic backtracking blows up.
* MRV / LCV / Forward-Checking each tame the explosion.
* Min-Conflicts scales nearly flat but is not complete.
* Every solver returns a best-partial assignment under load — the
  graceful-failure behavior the COP formulation demands.